In [ ]:
from typing import NamedTuple

import kfp
from kfp.components import func_to_container_op, InputPath, OutputPath


@func_to_container_op
def add_double(
        output_text_path: OutputPath(str), params1: int = 0, params2: int = 0):
    print("step in add_double")
    with open(output_text_path, 'w') as writer:
        writer.write(str(params1 + params2))


@func_to_container_op
def add_double_file(
        output_text_path: OutputPath(str), params1: InputPath(), params2: InputPath()):
    f1 = open(params1)
    number1 = int(f1.readline())
    f2 = open(params2)
    number2 = int(f2.readline())
    print("step in add_double_file")
    with open(output_text_path, 'w') as writer:
        writer.write(str(number1 + number2))
        
@func_to_container_op
def add_single(text_path: InputPath(),
               output_text_path: OutputPath(str),
               params1: int = 0):
    print("step in add_single")
    params2 = 0
    f = open(text_path)
    params2 = int(f.readline())
    with open(output_text_path, 'w') as writer:
        writer.write(str(params1 )+"\n")
        writer.write(str( params2)+"\n")
        writer.write(str(params1 + params2)+"\n")

        
three_params_option = kfp.components.load_component_from_url(
    'http://10.23.71.70:8088/notebook/kubeflow-user-example-com/gputest/files/pipelineTest/test1/component.yaml?_xsrf=2%7C38f023a1%7C32da0fa54686495ed8de40f76a2de5b6%7C1644913675')
    
def pipeline_my_handle():

    res1 = add_double(params1=10, params2=10)
    res2 = add_single(res1.output, params1=1)
    res3 = add_single(res1.output, params1=1)
    res4 = add_double_file(params1=res2.output, params2=res3.output)
    three_params_option(res4.output)
    

kfp.compiler.Compiler().compile(pipeline_my_handle, "caclulatewithremotecomponent" + '.yaml')